# CRLA School Performance Dashboard

Select a school using the filters below to view reading proficiency performance across assessment timepoints.

In [5]:
!ls "."

app.py	dashboard.ipynb  data  prepare_data.py


In [6]:
import pandas as pd
import numpy as np
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path

# ---------------------------------------------------------------------------
# Config
# ---------------------------------------------------------------------------

DATA_DIR = Path("data")

TIMEPOINT_ORDER = ["BoSY 2024-25", "EoSY 2024-25", "BoSY 2025-26"]

PROFILE_COLORS = {
    "Lower Emergent": "#DC143C",
    "Higher Emergent": "#FF8C00",
    "Developing": "#FFD700",
    "Transitioning": "#9ACD32",
    "Grade Level": "#228B22",
}

PROFILE_ORDER = [
    "Lower Emergent",
    "Higher Emergent",
    "Developing",
    "Transitioning",
    "Grade Level",
]

GRADE_LANG_ORDER = ["G1", "G2 MT", "G2 Fil", "G3 MT", "G3 Fil", "G3 Eng"]

SEGMENT_COLORS = {"Learning": "#4A90D9", "Retention": "#F5A623"}

# ---------------------------------------------------------------------------
# Load data
# ---------------------------------------------------------------------------

metadata = pd.read_parquet(DATA_DIR / "school_metadata.parquet")
profiles = pd.read_parquet(DATA_DIR / "school_profiles.parquet")
ordinal = pd.read_parquet(DATA_DIR / "school_ordinal.parquet")

print(f"Loaded: {len(metadata):,} schools, {len(profiles):,} profile rows, {len(ordinal):,} ordinal rows")

Loaded: 39,206 schools, 3,338,700 profile rows, 111,293 ordinal rows


In [7]:
# ---------------------------------------------------------------------------
# Helper functions
# ---------------------------------------------------------------------------

def get_segments(ordinal_school):
    """Compute segment deltas from a school's ordinal data."""
    tp_map = {tp: i for i, tp in enumerate(TIMEPOINT_ORDER)}
    tp_data = ordinal_school.copy()
    tp_data["tp_order"] = tp_data["timepoint_label"].map(tp_map)
    tp_data = tp_data.dropna(subset=["tp_order"]).sort_values("tp_order")

    segments = []
    for i in range(len(tp_data) - 1):
        row0 = tp_data.iloc[i]
        row1 = tp_data.iloc[i + 1]
        if not (row0["valid"] and row1["valid"]):
            continue

        sy0, p0 = row0["school_year"], row0["period"]
        sy1, p1 = row1["school_year"], row1["period"]

        if p0 == "BoSY" and p1 == "EoSY" and sy0 == sy1:
            seg_type, label = "Learning", f"Learning {sy0}"
        elif p0 == "EoSY" and p1 == "BoSY":
            seg_type, label = "Retention", f"Retention {sy0} \u2192 {sy1}"
        else:
            seg_type, label = "Other", f"{p0} {sy0} \u2192 {p1} {sy1}"

        segments.append({
            "label": label, "type": seg_type,
            "delta_overall": row1["ordinal_overall"] - row0["ordinal_overall"],
            "delta_G1": row1["ordinal_G1"] - row0["ordinal_G1"],
            "delta_G2": row1["ordinal_G2"] - row0["ordinal_G2"],
            "delta_G3": row1["ordinal_G3"] - row0["ordinal_G3"],
            "tp0": row0["timepoint_label"], "tp1": row1["timepoint_label"],
        })
    return segments

national_segments = get_segments(ordinal[ordinal["School ID"] == -1])


def build_profile_chart(school_id, timepoint_label, show_pct=True):
    """Build a horizontal stacked bar chart for one timepoint."""
    tp_data = profiles[
        (profiles["School ID"] == school_id) &
        (profiles["timepoint_label"] == timepoint_label)
    ].copy()

    if tp_data.empty:
        return None

    value_col = "percentage" if show_pct else "raw_count"
    x_label = "% of Learners" if show_pct else "Number of Learners"

    tp_data["grade_lang"] = pd.Categorical(
        tp_data["grade_lang"], categories=GRADE_LANG_ORDER, ordered=True
    )
    tp_data["profile"] = pd.Categorical(
        tp_data["profile"], categories=PROFILE_ORDER, ordered=True
    )
    tp_data = tp_data.sort_values(["grade_lang", "profile"])

    fig = px.bar(
        tp_data, y="grade_lang", x=value_col, color="profile",
        orientation="h",
        category_orders={"grade_lang": GRADE_LANG_ORDER, "profile": PROFILE_ORDER},
        color_discrete_map=PROFILE_COLORS,
        text=tp_data[value_col].apply(
            lambda v: f"{v:.1f}%" if show_pct and pd.notna(v)
            else (f"{int(v):,}" if pd.notna(v) else "")
        ),
        labels={value_col: x_label, "grade_lang": "Grade / Language", "profile": "Reading Profile"},
    )
    fig.update_layout(
        title=f"<b>{timepoint_label}</b>", title_font_size=16,
        height=260, margin=dict(l=0, r=0, t=40, b=0),
        legend=dict(orientation="h", yanchor="bottom", y=-0.4, xanchor="center", x=0.5, title=None),
        bargap=0.25,
        xaxis=dict(range=[0, 100] if show_pct else None),
    )
    fig.update_traces(textposition="inside", textfont_size=10)
    return fig


def build_trajectory_chart(school_id, school_name):
    """Build the ordinal score trajectory line chart."""
    tp_map = {tp: i for i, tp in enumerate(TIMEPOINT_ORDER)}

    school_traj = ordinal[
        (ordinal["School ID"] == school_id) &
        ordinal["timepoint_label"].isin(TIMEPOINT_ORDER) &
        ordinal["valid"]
    ].copy()
    school_traj["tp_order"] = school_traj["timepoint_label"].map(tp_map)
    school_traj = school_traj.sort_values("tp_order")

    nat_traj = ordinal[
        (ordinal["School ID"] == -1) &
        ordinal["timepoint_label"].isin(TIMEPOINT_ORDER)
    ].copy()
    nat_traj["tp_order"] = nat_traj["timepoint_label"].map(tp_map)
    nat_traj = nat_traj.sort_values("tp_order")

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=school_traj["timepoint_label"], y=school_traj["ordinal_overall"],
        mode="lines+markers+text", name=school_name,
        line=dict(color="#4A90D9", width=3), marker=dict(size=10),
        text=school_traj["ordinal_overall"].apply(lambda v: f"{v:.2f}"),
        textposition="top center", textfont=dict(size=12),
    ))
    fig.add_trace(go.Scatter(
        x=nat_traj["timepoint_label"], y=nat_traj["ordinal_overall"],
        mode="lines+markers+text", name="National Mean",
        line=dict(color="#999999", width=2, dash="dash"), marker=dict(size=8),
        text=nat_traj["ordinal_overall"].apply(lambda v: f"{v:.2f}"),
        textposition="bottom center", textfont=dict(size=11, color="#999999"),
    ))
    fig.update_layout(
        height=320, margin=dict(l=0, r=0, t=10, b=0),
        yaxis=dict(title="Ordinal Score", range=[1, 5], dtick=1),
        xaxis=dict(title=None),
        legend=dict(orientation="h", yanchor="bottom", y=-0.25, xanchor="center", x=0.5),
    )
    return fig


def build_grade_chart(school_segments):
    """Build the grade-level progress grouped bar chart."""
    grade_data = []
    for seg in school_segments:
        for grade in ["G1", "G2", "G3"]:
            grade_data.append({
                "Segment": seg["label"], "Grade": grade,
                "Delta": seg[f"delta_{grade}"],
            })
    grade_df = pd.DataFrame(grade_data)

    fig = px.bar(
        grade_df, x="Grade", y="Delta", color="Segment", barmode="group",
        color_discrete_map={
            seg["label"]: SEGMENT_COLORS.get(seg["type"], "#888888")
            for seg in school_segments
        },
        text=grade_df["Delta"].apply(lambda v: f"{v:+.2f}"),
        labels={"Delta": "Ordinal Score Change", "Grade": "Grade Level"},
    )
    fig.add_hline(y=0, line_dash="solid", line_color="#666666", line_width=1)
    fig.update_layout(
        height=350, margin=dict(l=0, r=0, t=10, b=0),
        legend=dict(orientation="h", yanchor="bottom", y=-0.25, xanchor="center", x=0.5, title=None),
    )
    fig.update_traces(textposition="outside", textfont_size=12)
    return fig


print("Helper functions ready.")

Helper functions ready.


In [8]:
# ---------------------------------------------------------------------------
# Dashboard UI
# ---------------------------------------------------------------------------

# --- Cascading filter widgets ---
regions = sorted(metadata["Region"].dropna().unique())
w_region = widgets.Dropdown(options=["All"] + regions, value="All", description="Region:")
w_division = widgets.Dropdown(options=["All"], value="All", description="Division:")
w_district = widgets.Dropdown(options=["All"], value="All", description="District:")
w_school = widgets.Dropdown(options=[], description="School:", style={"description_width": "60px"})
w_show_pct = widgets.ToggleButtons(options=["Percentage", "Raw Count"], value="Percentage", description="Show as:")

# Output areas
out_header = widgets.Output()
out_profiles = widgets.Output()
out_progress = widgets.Output()
out_grades = widgets.Output()

def update_divisions(*args):
    filtered = metadata if w_region.value == "All" else metadata[metadata["Region"] == w_region.value]
    divs = sorted(filtered["Division"].dropna().unique())
    w_division.options = ["All"] + divs
    w_division.value = "All"

def update_districts(*args):
    filtered = metadata if w_region.value == "All" else metadata[metadata["Region"] == w_region.value]
    if w_division.value != "All":
        filtered = filtered[filtered["Division"] == w_division.value]
    dists = sorted(filtered["District"].dropna().unique())
    w_district.options = ["All"] + dists
    w_district.value = "All"

def update_schools(*args):
    filtered = metadata.copy()
    if w_region.value != "All":
        filtered = filtered[filtered["Region"] == w_region.value]
    if w_division.value != "All":
        filtered = filtered[filtered["Division"] == w_division.value]
    if w_district.value != "All":
        filtered = filtered[filtered["District"] == w_district.value]
    filtered = filtered.sort_values("School Name")
    options = [(f"{row['School ID']} — {row['School Name']}", row["School ID"]) for _, row in filtered.iterrows()]
    w_school.options = options
    w_school.value = options[0][1] if options else None

def render_dashboard(*args):
    school_id = w_school.value
    if school_id is None:
        return

    school_meta = metadata[metadata["School ID"] == school_id].iloc[0]
    school_ord = ordinal[ordinal["School ID"] == school_id]
    show_pct = w_show_pct.value == "Percentage"

    # --- Section A: Header ---
    with out_header:
        clear_output(wait=True)
        display(HTML(
            f"<h2>{school_meta['School Name']}</h2>"
            f"<p><b>{school_meta['Division']}</b> / {school_meta['District']} · "
            f"<em>{school_meta['Region']}</em></p>"
        ))
        available_tps = [tp for tp in TIMEPOINT_ORDER if tp in school_ord["timepoint_label"].values]
        if available_tps:
            header_parts = []
            for tp in available_tps:
                tp_row = school_ord[school_ord["timepoint_label"] == tp].iloc[0]
                total = tp_row["total_assessed"]
                total_str = f"{int(total):,}" if pd.notna(total) else "—"
                header_parts.append(
                    f"<div style='display:inline-block; text-align:center; margin-right:40px;'>"
                    f"<div style='font-size:13px; color:#666;'>{tp}</div>"
                    f"<div style='font-size:28px; font-weight:bold;'>{total_str}</div>"
                    f"<div style='font-size:12px; color:#999;'>assessed</div></div>"
                )
            display(HTML("<div style='margin:10px 0 20px 0;'>" + "".join(header_parts) + "</div>"))

    # --- Section B: Reading Profile Distribution ---
    with out_profiles:
        clear_output(wait=True)
        display(HTML("<h3>Reading Profile Distribution</h3>"))
        available_tps = [tp for tp in TIMEPOINT_ORDER if tp in school_ord["timepoint_label"].values]
        for tp in available_tps:
            fig = build_profile_chart(school_id, tp, show_pct=show_pct)
            if fig:
                fig.show()

    # --- Section C: Ordinal Progress Score ---
    with out_progress:
        clear_output(wait=True)
        display(HTML("<h3>Ordinal Progress Score</h3>"))
        school_segments = get_segments(school_ord)

        if not school_segments:
            display(HTML("<p style='color:#999;'>Not enough valid timepoints to compute progress.</p>"))
        else:
            # Metric cards
            card_parts = []
            for seg in school_segments:
                nat_delta = None
                for ns in national_segments:
                    if ns["tp0"] == seg["tp0"] and ns["tp1"] == seg["tp1"]:
                        nat_delta = ns["delta_overall"]
                        break
                vs_nat = seg["delta_overall"] - nat_delta if nat_delta is not None else None
                color = "#228B22" if seg["delta_overall"] >= 0 else "#DC143C"
                vs_color = "#228B22" if vs_nat and vs_nat >= 0 else "#DC143C"
                vs_str = f"<div style='font-size:12px; color:{vs_color};'>{vs_nat:+.3f} vs national ({nat_delta:+.3f})</div>" if vs_nat is not None else ""
                card_parts.append(
                    f"<div style='display:inline-block; text-align:center; margin-right:50px; "
                    f"padding:15px 25px; border:1px solid #ddd; border-radius:8px;'>"
                    f"<div style='font-size:13px; color:#666; margin-bottom:5px;'>{seg['label']}</div>"
                    f"<div style='font-size:32px; font-weight:bold; color:{color};'>{seg['delta_overall']:+.3f}</div>"
                    f"{vs_str}</div>"
                )
            display(HTML("<div style='margin:15px 0;'>" + "".join(card_parts) + "</div>"))

            # Trajectory chart
            display(HTML("<h4>Score Trajectory</h4>"))
            fig_traj = build_trajectory_chart(school_id, school_meta["School Name"])
            fig_traj.show()

            # Interpretation
            display(HTML(
                "<details style='margin-top:10px;'><summary style='cursor:pointer; color:#4A90D9;'>"
                "How to read the ordinal score</summary>"
                "<p style='font-size:13px; margin-top:8px;'>"
                "The ordinal score is the <b>average reading proficiency level</b> (1-5 scale):<br>"
                "1 = Lower Emergent, 2 = Higher Emergent, 3 = Developing, 4 = Transitioning, 5 = Grade Level.<br><br>"
                "A <b>positive delta</b> means students moved up in proficiency. "
                "A <b>negative delta</b> means students moved down.<br>"
                "The 'vs national' shows whether the school's change was better or worse than the national average."
                "</p></details>"
            ))

    # --- Section D: Progress by Grade Level ---
    with out_grades:
        clear_output(wait=True)
        display(HTML("<h3>Progress by Grade Level</h3>"))
        school_segments = get_segments(school_ord)
        if not school_segments:
            display(HTML("<p style='color:#999;'>Not enough valid timepoints.</p>"))
        else:
            fig_grade = build_grade_chart(school_segments)
            fig_grade.show()
            display(HTML(
                "<p style='font-size:12px; color:#888;'>"
                "Bars above zero = improvement; bars below zero = decline. "
                "Each grade's score is the mean ordinal proficiency across its language groups.</p>"
            ))


# --- Wire up observers ---
w_region.observe(update_divisions, names="value")
w_division.observe(update_districts, names="value")
w_district.observe(update_schools, names="value")
w_school.observe(render_dashboard, names="value")
w_show_pct.observe(render_dashboard, names="value")

# Initialize
update_divisions()
update_districts()
update_schools()

# --- Layout ---
filter_box = widgets.VBox([
    widgets.HTML("<h3>Filters</h3>"),
    w_region, w_division, w_district, w_school,
    widgets.HTML("<hr>"),
    w_show_pct,
], layout=widgets.Layout(padding="10px", border="1px solid #ddd", border_radius="8px", width="100%"))

dashboard = widgets.VBox([
    filter_box,
    out_header,
    out_profiles,
    out_progress,
    out_grades,
], layout=widgets.Layout(max_width="1000px"))

display(dashboard)

# Trigger initial render
render_dashboard()